Загрузка датасета
random seed 
sanity check

In [ ]:
import random
from typing import List, Dict, Any

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision, torch
from datasets import Dataset
from datasets import load_dataset
import numpy as np
from sklearn.model_selection import train_test_split

from torchvision import transforms
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    pipeline,
)
from datasets import load_dataset
import numpy as np
from sklearn.model_selection import train_test_split

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

разбивка на трейн вал тест

In [ ]:

dataset = load_dataset("emotion")

print(f"Размер датасета: {len(dataset['train']) + len(dataset['test'])}")

class_names = dataset['train'].features['label'].names
print(f"\nНазвания классов:")
for i, name in enumerate(class_names):
    print(f"  {i}: {name}")

print("\nПримеры текста:")
for i in range(5):
    text = dataset['train'][i]['text']
    label = dataset['train'][i]['label']
    print(f"  Пример {i+1}:")
    print(f"    Текст: {text[:150]}...")
    print(f"    Метка: {label} - {class_names[label]}")

all_texts = list(dataset['train']['text']) + list(dataset['test']['text'])
all_labels = list(dataset['train']['label']) + list(dataset['test']['label'])

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    all_texts, all_labels,
    test_size=0.2,
    stratify=all_labels
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels,
    test_size=0.5,
    stratify=temp_labels
)

train_dataset = Dataset.from_dict({'text': train_texts, 'label': train_labels})
val_dataset = Dataset.from_dict({'text': val_texts, 'label': val_labels})
test_dataset = Dataset.from_dict({'text': test_texts, 'label': test_labels})

print(f"\n  Train:      {len(train_dataset)}")
print(f"  Validation: {len(val_dataset)}")
print(f"  Test:       {len(test_dataset)}")
print(f"  Total:      {len(all_texts)} ")


Размер датасета: 18000

Названия классов:
  0: sadness
  1: joy
  2: love
  3: anger
  4: fear
  5: surprise

Примеры текста:
  Пример 1:
    Текст: i didnt feel humiliated...
    Метка: 0 - sadness
  Пример 2:
    Текст: i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake...
    Метка: 0 - sadness
  Пример 3:
    Текст: im grabbing a minute to post i feel greedy wrong...
    Метка: 3 - anger
  Пример 4:
    Текст: i am ever feeling nostalgic about the fireplace i will know that it is still on the property...
    Метка: 2 - love
  Пример 5:
    Текст: i am feeling grouchy...
    Метка: 3 - anger

  Train:      14400
  Validation: 1800
  Test:       1800
  Total:      18000 


классифицируется текст по настроению
каждому тексту присваивается метка в зависимости от определенной эмоции

In [ ]:

MODEL_NAME = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

batch_no_trunc = tokenizer(
    [short_text, long_text],
    padding=True,
    return_tensors="pt",
)

batch_with_trunc = tokenizer(
    [short_text, long_text],
    padding=True,
    truncation=True,
    max_length=16,
    return_tensors="pt",
)

print("Без truncation:")
print("input_ids shape:", tuple(batch_no_trunc["input_ids"].shape))
print("attention_mask shape:", tuple(batch_no_trunc["attention_mask"].shape))
print()

print("С truncation и max_length=16:")
print("input_ids shape:", tuple(batch_with_trunc["input_ids"].shape))
print("attention_mask shape:", tuple(batch_with_trunc["attention_mask"].shape))
print()

for idx, text in enumerate([short_text, long_text]):
    print(f"--- Пример {idx} ---")
    print("Исходный текст:", text)
    print("Токены после truncation:")
    print(tokenizer.convert_ids_to_tokens(batch_with_trunc["input_ids"][idx].tolist()))
    print("attention_mask:")
    print(batch_with_trunc["attention_mask"][idx].tolist())
    print()

# Покажем padding в табличном виде на примере короткого текста из batch_no_trunc.
# Здесь хорошо видно: [PAD]-токены стоят после [SEP] и получают attention_mask=0.
pad_ids = batch_no_trunc["input_ids"][0].tolist()
pad_tokens = tokenizer.convert_ids_to_tokens(pad_ids)
pad_mask = batch_no_trunc["attention_mask"][0].tolist()

print("Короткий текст в батче с padding (batch_no_trunc, пример 0):")
display(
    pd.DataFrame(
        {
            "position": list(range(len(pad_ids))),
            "token": pad_tokens,
            "token_id": pad_ids,
            "attention_mask": pad_mask,
        }
    )
)
print("Позиции с [PAD] и attention_mask=0 – модель их игнорирует.")